# Recipe Recommender System
### CSC 577: Recommender Systems — Winter 2026 — DePaul University
**Team Rocket:** Mohammed Ahmed Hussain, Sameer Ahmed Noor, Drishti Arora

**Repository:** [github.com/totallyahmed/Recipe-Recommender](https://github.com/totallyahmed/Recipe-Recommender)

This notebook builds a hybrid recipe recommender from scratch. Two collaborative filtering algorithms — user-based CF and SVD matrix factorization — are paired with a constraint layer that checks ingredient availability. Everything uses NumPy and SciPy directly; no recommendation libraries.

**Data:** [Food.com Recipes & Interactions](https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions) — place `RAW_recipes.csv` and `RAW_interactions.csv` in the `data/` folder before running.

## Table of Contents
1. [Imports and Configuration](#1)
2. [Load Raw Data](#2)
3. [Exploratory Data Analysis](#3)
4. [Preprocessing](#4)
5. [Train/Test Split and Sparse Matrix](#5)
6. [SVD Matrix Factorization](#6)
7. [User-Based Collaborative Filtering](#7)
8. [Hyperparameter Tuning](#8)
9. [Top-N Candidate Generation](#9)
10. [Popularity Baseline](#10)
11. [Ingredient Constraint Filtering](#11)
12. [Constraint Threshold Analysis](#12)
13. [Evaluation](#13)
14. [User-Stratified Analysis](#14)
15. [Summary](#15)

<a id='1'></a>
## 1. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from sklearn.model_selection import train_test_split
from collections import Counter
import warnings, os, gc, time, ast

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)

os.makedirs('data/processed', exist_ok=True)
os.makedirs('results', exist_ok=True)

# ── Parameters ──
MIN_USER_RATINGS   = 10    # minimum ratings per user
MIN_RECIPE_RATINGS = 5     # minimum ratings per recipe
TEST_SIZE          = 0.2   # hold-out fraction per user
RELEVANCE_THRESHOLD = 4    # rating >= 4 counts as "liked"
TOP_N              = 20    # candidates per user
K_VALUES           = [5, 10, 20]

SVD_K              = 50    # default latent factors
UBCF_K             = 50    # default neighborhood size
SOFT_MAX_MISSING   = 3     # soft constraint threshold
PANTRY_COVERAGE    = 0.7   # fraction of user's ingredients kept

print('Libraries loaded.')

<a id='2'></a>
## 2. Load Raw Data

In [ ]:
recipes_raw = pd.read_csv('data/RAW_recipes.csv')
interactions_raw = pd.read_csv('data/RAW_interactions.csv')

print(f'Recipes:      {len(recipes_raw):,}')
print(f'Interactions: {len(interactions_raw):,}')
print(f'Users:        {interactions_raw["user_id"].nunique():,}')
print(f'Recipes:      {interactions_raw["recipe_id"].nunique():,}')
recipes_raw.head(3)

<a id='3'></a>
## 3. Exploratory Data Analysis

### 3.1 Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(interactions_raw['rating'], bins=range(0, 7), align='left',
             color='steelblue', edgecolor='white', rwidth=0.8)
axes[0].set_title('Rating Distribution (including 0)')
axes[0].set_xlabel('Rating'); axes[0].set_ylabel('Count')

valid = interactions_raw[interactions_raw['rating'] > 0]
axes[1].hist(valid['rating'], bins=range(1, 7), align='left',
             color='coral', edgecolor='white', rwidth=0.8)
axes[1].set_title('Rating Distribution (valid ratings only)')
axes[1].set_xlabel('Rating'); axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('results/rating_distribution.png', bbox_inches='tight')
plt.show()

n_zeros = (interactions_raw['rating'] == 0).sum()
print(f'Zero-value ratings: {n_zeros:,} ({n_zeros/len(interactions_raw):.1%})')
print(f'Ratings 4 or 5: {(valid["rating"] >= 4).mean():.1%} of valid ratings')

### 3.2 User Activity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

user_counts = valid.groupby('user_id').size()
axes[0].hist(user_counts, bins=100, color='steelblue', edgecolor='white', log=True)
axes[0].set_title('Ratings per User (log scale)')
axes[0].set_xlabel('Number of Ratings'); axes[0].set_ylabel('Number of Users (log)')

axes[1].hist(user_counts[user_counts <= 100], bins=100, color='coral', edgecolor='white')
axes[1].set_title('Ratings per User (≤ 100 ratings, zoomed)')
axes[1].set_xlabel('Number of Ratings'); axes[1].set_ylabel('Number of Users')

plt.tight_layout()
plt.savefig('results/user_activity.png', bbox_inches='tight')
plt.show()
print(f'Median: {user_counts.median():.0f} | Mean: {user_counts.mean():.1f}')

### 3.3 Recipe Popularity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

recipe_counts = valid.groupby('recipe_id').size()
axes[0].hist(recipe_counts, bins=100, color='green', edgecolor='white', log=True)
axes[0].set_title('Ratings per Recipe (log scale)')
axes[0].set_xlabel('Number of Ratings'); axes[0].set_ylabel('Number of Recipes (log)')

axes[1].hist(recipe_counts[recipe_counts <= 50], bins=50, color='purple', edgecolor='white')
axes[1].set_title('Ratings per Recipe (≤ 50 ratings, zoomed)')
axes[1].set_xlabel('Number of Ratings'); axes[1].set_ylabel('Number of Recipes')

plt.tight_layout()
plt.savefig('results/recipe_popularity.png', bbox_inches='tight')
plt.show()

### 3.4 Average Recipe Ratings

In [ ]:
avg_ratings = valid.groupby('recipe_id')['rating'].mean()

plt.figure(figsize=(10, 4))
plt.hist(avg_ratings, bins=50, color='coral', edgecolor='white')
plt.title('Distribution of Average Recipe Ratings')
plt.xlabel('Average Rating'); plt.ylabel('Number of Recipes')
plt.tight_layout()
plt.savefig('results/avg_recipe_ratings.png', bbox_inches='tight')
plt.show()

print(f'Perfect 5.0 average: {(avg_ratings == 5.0).sum():,} recipes ({(avg_ratings == 5.0).mean():.1%})')

### 3.5 Ingredients

In [ ]:
recipes_raw['ingredient_list'] = recipes_raw['ingredients'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_ings = recipes_raw['ingredient_list'].apply(len)
axes[0].hist(n_ings[n_ings > 0], bins=range(1, 45), color='orange', edgecolor='white')
axes[0].set_title('Number of Ingredients per Recipe')
axes[0].set_xlabel('Ingredients'); axes[0].set_ylabel('Recipes')

all_ings = Counter()
for ings in recipes_raw['ingredient_list']:
    all_ings.update(ings)
top20 = all_ings.most_common(20)
names, counts = zip(*top20)
axes[1].barh(range(len(names)), counts, color='orange', edgecolor='white')
axes[1].set_yticks(range(len(names))); axes[1].set_yticklabels(names)
axes[1].invert_yaxis()
axes[1].set_title('Top 20 Most Common Ingredients')
axes[1].set_xlabel('Number of Recipes')

plt.tight_layout()
plt.savefig('results/ingredient_analysis.png', bbox_inches='tight')
plt.show()

### 3.6 Tags

In [ ]:
recipes_raw['tag_list'] = recipes_raw['tags'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

tag_counter = Counter()
for tags in recipes_raw['tag_list']:
    tag_counter.update(tags)

top25 = tag_counter.most_common(25)
tn, tc = zip(*top25)

plt.figure(figsize=(10, 7))
plt.barh(range(len(tn)), tc, color='steelblue', edgecolor='white')
plt.yticks(range(len(tn)), tn); plt.gca().invert_yaxis()
plt.title('Top 25 Most Common Recipe Tags')
plt.xlabel('Number of Recipes')
plt.tight_layout()
plt.savefig('results/tag_analysis.png', bbox_inches='tight')
plt.show()

### 3.7 Cooking Time

In [ ]:
filtered_minutes = recipes_raw[(recipes_raw['minutes'] > 0) & (recipes_raw['minutes'] <= 300)]

plt.figure(figsize=(12, 4))
plt.hist(filtered_minutes['minutes'], bins=60, color='mediumseagreen', edgecolor='white')
plt.title('Cooking Time Distribution (1–300 minutes)')
plt.xlabel('Minutes'); plt.ylabel('Number of Recipes')
plt.tight_layout()
plt.savefig('results/cooking_time.png', bbox_inches='tight')
plt.show()
print(f'Median cooking time: {filtered_minutes["minutes"].median():.0f} min')

<a id='4'></a>
## 4. Preprocessing

1. Drop zero-value ratings (recording artifacts on Food.com, not real dislikes)
2. Deduplicate — keep most recent if a user rated the same recipe twice
3. Iteratively filter: users with ≥ 10 ratings, recipes with ≥ 5 ratings
4. Parse ingredient lists, clean metadata, handle outlier cook times
5. Re-encode IDs as contiguous integers for sparse matrix indexing

In [ ]:
# 4a. Drop zeros
interactions = interactions_raw[interactions_raw['rating'] > 0].copy()
print(f'Dropped {len(interactions_raw) - len(interactions):,} zero-value ratings')

# 4b. Deduplicate
interactions['date'] = pd.to_datetime(interactions['date'])
interactions = interactions.sort_values('date', ascending=False)
before = len(interactions)
interactions = interactions.drop_duplicates(subset=['user_id', 'recipe_id'], keep='first')
print(f'Dropped {before - len(interactions):,} duplicate interactions')
print(f'Remaining: {len(interactions):,}')

In [ ]:
# 4c. Iterative filtering
def iterative_filter(df, min_user, min_recipe, max_iter=10):
    for iteration in range(max_iter):
        before = len(df)
        uc = df['user_id'].value_counts()
        df = df[df['user_id'].isin(uc[uc >= min_user].index)]
        rc = df['recipe_id'].value_counts()
        df = df[df['recipe_id'].isin(rc[rc >= min_recipe].index)]
        if len(df) == before:
            print(f'  Converged after {iteration + 1} iterations')
            break
    return df

interactions = iterative_filter(interactions, MIN_USER_RATINGS, MIN_RECIPE_RATINGS)

print(f'After filtering:')
print(f'  Interactions: {len(interactions):,}')
print(f'  Users:        {interactions["user_id"].nunique():,}')
print(f'  Recipes:      {interactions["recipe_id"].nunique():,}')

In [ ]:
# 4d. Clean recipe metadata
valid_ids = interactions['recipe_id'].unique()
recipes = recipes_raw[recipes_raw['id'].isin(valid_ids)].copy()
recipes = recipes.rename(columns={'id': 'recipe_id'})

def safe_parse(x):
    try:
        result = ast.literal_eval(x)
        return result if isinstance(result, list) else []
    except:
        return []

# Parse ingredients into a pipe-separated lowercase string
recipes['ingredients_parsed'] = recipes['ingredients'].apply(safe_parse)
recipes['ingredients_str'] = recipes['ingredients_parsed'].apply(
    lambda x: '|'.join(i.strip().lower() for i in x))
recipes['n_ingredients'] = recipes['ingredients_parsed'].apply(len)

# Parse nutrition: [calories, total_fat_%DV, sugar_%DV, sodium_%DV, protein_%DV, sat_fat_%DV, carbs_%DV]
nutrition_cols = ['calories', 'total_fat_pdv', 'sugar_pdv', 'sodium_pdv',
                  'protein_pdv', 'sat_fat_pdv', 'carbs_pdv']
recipes['nutrition_parsed'] = recipes['nutrition'].apply(safe_parse)
for i, col in enumerate(nutrition_cols):
    recipes[col] = recipes['nutrition_parsed'].apply(
        lambda x, idx=i: float(x[idx]) if len(x) > idx else np.nan)

# Drop outlier cook times
recipes = recipes[(recipes['minutes'] > 0) & (recipes['minutes'] <= 10080)]

print(f'Recipes after cleaning: {len(recipes):,}')

In [ ]:
# 4e. Contiguous ID mappings
unique_users   = sorted(interactions['user_id'].unique())
unique_recipes = sorted(interactions['recipe_id'].unique())

user2idx   = {u: i for i, u in enumerate(unique_users)}
recipe2idx = {r: i for i, r in enumerate(unique_recipes)}
idx2user   = {i: u for u, i in user2idx.items()}
idx2recipe = {i: r for r, i in recipe2idx.items()}

interactions = interactions.copy()
interactions['u_idx'] = interactions['user_id'].map(user2idx)
interactions['r_idx'] = interactions['recipe_id'].map(recipe2idx)

n_users   = len(unique_users)
n_recipes = len(unique_recipes)
sparsity  = 1 - len(interactions) / (n_users * n_recipes)

print(f'Matrix:   {n_users:,} users × {n_recipes:,} recipes')
print(f'Ratings:  {len(interactions):,}')
print(f'Sparsity: {sparsity:.4%}')

In [ ]:
# 4f. Save processed data
interactions.to_csv('data/processed/interactions_clean.csv', index=False)

recipes_out = recipes[['recipe_id', 'name', 'minutes', 'contributor_id', 'submitted',
    'tags', 'n_steps', 'description', 'ingredients_str', 'n_ingredients'] + nutrition_cols].copy()
recipes_out = recipes_out.rename(columns={'ingredients_str': 'ingredients'})
recipes_out.to_csv('data/processed/recipes_clean.csv', index=False)

# ID mappings
pd.DataFrame({'user_id': unique_users, 'u_idx': range(n_users)}).to_csv(
    'data/processed/user_mapping.csv', index=False)
pd.DataFrame({'recipe_id': unique_recipes, 'r_idx': range(n_recipes)}).to_csv(
    'data/processed/recipe_mapping.csv', index=False)

print('Saved to data/processed/')

<a id='5'></a>
## 5. Train/Test Split and Sparse Matrix

Hold out 20% of each user's ratings. Users with fewer than 5 ratings go entirely into training.

The sparse representation keeps memory at a few MB instead of the ~2.7 GB a dense matrix would need.

In [ ]:
def train_test_split_per_user(df, test_size=0.2, random_state=42):
    """Stratified split: hold out test_size of each user's ratings."""
    train_parts, test_parts = [], []
    for _, group in df.groupby('user_id'):
        if len(group) < 5:
            train_parts.append(group)
            continue
        tr, te = train_test_split(group, test_size=test_size, random_state=random_state)
        train_parts.append(tr)
        test_parts.append(te)
    return (pd.concat(train_parts).reset_index(drop=True),
            pd.concat(test_parts).reset_index(drop=True))

train_df, test_df = train_test_split_per_user(interactions)

train_df.to_csv('data/processed/train.csv', index=False)
test_df.to_csv('data/processed/test.csv', index=False)

print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')

In [ ]:
# Build sparse rating matrix from training data
R_sparse = csr_matrix(
    (train_df['rating'].values.astype(np.float64),
     (train_df['u_idx'].values, train_df['r_idx'].values)),
    shape=(n_users, n_recipes))

# Per-user mean ratings
row_sums   = np.array(R_sparse.sum(axis=1)).flatten()
row_counts = np.array((R_sparse > 0).sum(axis=1)).flatten()
user_means = np.where(row_counts > 0, row_sums / row_counts, 0.0)

# Mean-center the sparse matrix
R_centered = R_sparse.copy().astype(np.float64)
nz_rows, nz_cols = R_centered.nonzero()
for i in range(len(nz_rows)):
    R_centered[nz_rows[i], nz_cols[i]] -= user_means[nz_rows[i]]
R_centered = csr_matrix(R_centered)
del nz_rows, nz_cols; gc.collect()

print(f'Sparse matrix: {R_sparse.shape}, {R_sparse.nnz:,} entries')
print(f'Memory: {R_sparse.data.nbytes / 1e6:.1f} MB sparse vs ~{n_users * n_recipes * 8 / 1e9:.1f} GB dense')

In [ ]:
# Map test entries to matrix indices
test_mapped = test_df.copy()
test_mapped = test_mapped.dropna(subset=['u_idx', 'r_idx'])
test_mapped[['u_idx', 'r_idx']] = test_mapped[['u_idx', 'r_idx']].astype(int)
print(f'Test entries mapped: {len(test_mapped):,}')

<a id='6'></a>
## 6. SVD Matrix Factorization

Truncated SVD on the mean-centered sparse matrix: $R \approx U\Sigma V^\top$

Predictions for user $u$, recipe $i$:
$$\hat{r}(u, i) = \mathbf{u}_u \cdot \Sigma \cdot \mathbf{v}_i^\top + \bar{r}(u)$$

We run `scipy.sparse.linalg.svds` directly on the sparse matrix — no dense intermediate.

In [ ]:
t0 = time.time()
U, sigma, Vt = svds(R_centered, k=SVD_K)
# svds returns factors in ascending singular value order — flip to descending
U, sigma, Vt = U[:, ::-1], sigma[::-1], Vt[::-1, :]
U_sigma = U * sigma  # precompute for fast prediction

print(f'SVD (k={SVD_K}) done in {time.time() - t0:.1f}s')
print(f'U: {U.shape} | sigma: {sigma.shape} | Vt: {Vt.shape}')

In [ ]:
# Scree plot + cumulative variance
cumvar = np.cumsum(sigma ** 2) / (sigma ** 2).sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sigma, 'o-', color='steelblue', markersize=4)
axes[0].set_title('Singular Values (Scree Plot)')
axes[0].set_xlabel('Factor index'); axes[0].set_ylabel('Singular value')

axes[1].plot(cumvar * 100, 'o-', color='coral', markersize=4)
axes[1].axhline(80, color='gray', ls='--', label='80%')
axes[1].set_title('Cumulative Variance Explained')
axes[1].set_xlabel('Number of factors'); axes[1].set_ylabel('%')
axes[1].legend()

plt.tight_layout()
plt.savefig('results/svd_variance.png', bbox_inches='tight')
plt.show()
print(f'{SVD_K} factors capture {cumvar[-1] * 100:.1f}% of variance')

In [ ]:
# SVD predictions on test set (vectorized)
svd_preds = np.sum(
    U_sigma[test_mapped['u_idx'].values] * Vt[:, test_mapped['r_idx'].values].T,
    axis=1) + user_means[test_mapped['u_idx'].values]
svd_preds = np.clip(svd_preds, 1.0, 5.0)
test_mapped['svd_pred'] = svd_preds

svd_rmse = np.sqrt(np.mean((test_mapped['rating'].values - svd_preds) ** 2))
svd_mae  = np.mean(np.abs(test_mapped['rating'].values - svd_preds))
print(f'SVD: RMSE = {svd_rmse:.4f}, MAE = {svd_mae:.4f}')

<a id='7'></a>
## 7. User-Based Collaborative Filtering

$$\hat{r}(u, i) = \bar{r}(u) + \frac{\sum_v \text{sim}(u,v) \cdot (r(v,i) - \bar{r}(v))}{\sum_v |\text{sim}(u,v)|}$$

Cosine similarity on the mean-centered sparse matrix. Similarity is computed per-user on demand so we never materialize the full 9K × 9K similarity matrix.

In [ ]:
def predict_ubcf(test_df, Rc, Rs, umeans, k_neighbors=50, sample_size=None):
    """
    Sparse UB-CF: for each test user, compute similarity against all users,
    pick top-k neighbors who rated the target recipe, predict.
    """
    if sample_size and sample_size < len(test_df):
        np.random.seed(42)
        test_df = test_df.iloc[
            np.random.choice(len(test_df), sample_size, replace=False)
        ].copy().reset_index(drop=True)

    preds = np.full(len(test_df), np.nan)
    t0 = time.time()

    for count, (u_idx, grp) in enumerate(test_df.groupby('u_idx')):
        if count % 1000 == 0 and count > 0:
            print(f'  {count} users done ({time.time() - t0:.0f}s)')

        sim = sk_cosine(Rc[u_idx], Rc).flatten()
        sim[u_idx] = 0
        sim = np.maximum(sim, 0)  # only positive similarities

        for idx in grp.index:
            r_idx = test_df.loc[idx, 'r_idx']

            # who rated this recipe?
            raters = np.array(Rs[:, r_idx].toarray()).flatten()
            rater_mask = raters > 0

            if not rater_mask.any():
                preds[idx] = umeans[u_idx]
                continue

            neighbor_sims = sim * rater_mask

            # keep only top-k
            if k_neighbors < len(sim):
                topk = np.argsort(neighbor_sims)[-k_neighbors:]
                mask = np.zeros(len(sim), dtype=bool)
                mask[topk] = True
                neighbor_sims = neighbor_sims * mask

            denom = neighbor_sims.sum()
            if denom == 0:
                preds[idx] = umeans[u_idx]
                continue

            centered = np.array(Rc[:, r_idx].toarray()).flatten()
            preds[idx] = np.clip(umeans[u_idx] + (neighbor_sims * centered).sum() / denom, 1.0, 5.0)

    test_df = test_df.copy()
    test_df['ubcf_pred'] = preds
    valid = (~np.isnan(preds)).sum()
    print(f'  Done: {valid}/{len(test_df)} predictions in {time.time() - t0:.0f}s')
    return test_df

In [ ]:
print(f'Running UB-CF on {len(test_mapped):,} test entries (k={UBCF_K})...')

ubcf_result = predict_ubcf(
    test_mapped[['u_idx', 'r_idx', 'rating', 'user_id', 'recipe_id']].copy(),
    R_centered, R_sparse, user_means, k_neighbors=UBCF_K)

test_mapped['ubcf_pred'] = ubcf_result['ubcf_pred'].values
vm = ~np.isnan(test_mapped['ubcf_pred'])

ubcf_rmse = np.sqrt(np.mean((test_mapped.loc[vm, 'rating'].values - test_mapped.loc[vm, 'ubcf_pred'].values) ** 2))
ubcf_mae  = np.mean(np.abs(test_mapped.loc[vm, 'rating'].values - test_mapped.loc[vm, 'ubcf_pred'].values))
print(f'\nUB-CF: RMSE = {ubcf_rmse:.4f}, MAE = {ubcf_mae:.4f}')

# Save test set with both model predictions
test_mapped.to_csv('data/processed/test_with_predictions.csv', index=False)
print('Saved test_with_predictions.csv')

<a id='8'></a>
## 8. Hyperparameter Tuning

### 8a. SVD — Latent Factor Count

Sweep k from 10 to 150. Each decomposition runs in 1–4 seconds on the sparse matrix.

In [ ]:
svd_tune = []

for k in [10, 20, 30, 50, 75, 100, 150]:
    t0 = time.time()
    Uk, sk, Vtk = svds(R_centered, k=k)
    Uk, sk, Vtk = Uk[:, ::-1], sk[::-1], Vtk[::-1, :]

    p = np.clip(
        np.sum((Uk * sk)[test_mapped['u_idx'].values] * Vtk[:, test_mapped['r_idx'].values].T, axis=1)
        + user_means[test_mapped['u_idx'].values], 1, 5)
    r = np.sqrt(np.mean((test_mapped['rating'].values - p) ** 2))
    m = np.mean(np.abs(test_mapped['rating'].values - p))

    svd_tune.append({'k': k, 'RMSE': r, 'MAE': m})
    print(f'  k={k:3d}: RMSE={r:.4f}  MAE={m:.4f}  ({time.time() - t0:.1f}s)')
    del Uk, sk, Vtk; gc.collect()

svd_tune_df = pd.DataFrame(svd_tune)
best_svd_k = int(svd_tune_df.loc[svd_tune_df['RMSE'].idxmin(), 'k'])
print(f'\nBest k = {best_svd_k}, RMSE = {svd_tune_df["RMSE"].min():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(svd_tune_df['k'], svd_tune_df['RMSE'], 'o-', color='steelblue', lw=2)
axes[0].set_title('SVD: RMSE vs k'); axes[0].set_xlabel('k'); axes[0].set_ylabel('RMSE')
axes[1].plot(svd_tune_df['k'], svd_tune_df['MAE'], 'o-', color='coral', lw=2)
axes[1].set_title('SVD: MAE vs k'); axes[1].set_xlabel('k'); axes[1].set_ylabel('MAE')
plt.tight_layout(); plt.savefig('results/svd_tuning.png', bbox_inches='tight'); plt.show()

### 8b. UB-CF — Neighborhood Size

Sweep k ∈ {10, 20, 50, 100, 200} on a 3,000-entry test sample (full evaluation takes several minutes per setting).

In [ ]:
ubcf_tune = []

for kn in [10, 20, 50, 100, 200]:
    res = predict_ubcf(
        test_mapped[['u_idx', 'r_idx', 'rating', 'user_id', 'recipe_id']].copy(),
        R_centered, R_sparse, user_means, k_neighbors=kn, sample_size=3000)
    v = ~np.isnan(res['ubcf_pred'])
    r = np.sqrt(np.mean((res.loc[v, 'rating'].values - res.loc[v, 'ubcf_pred'].values) ** 2))
    m = np.mean(np.abs(res.loc[v, 'rating'].values - res.loc[v, 'ubcf_pred'].values))
    ubcf_tune.append({'k': kn, 'RMSE': r, 'MAE': m})
    print(f'  k={kn:3d}: RMSE={r:.4f}  MAE={m:.4f}')

ubcf_tune_df = pd.DataFrame(ubcf_tune)
best_ubcf_k = int(ubcf_tune_df.loc[ubcf_tune_df['RMSE'].idxmin(), 'k'])
print(f'\nBest k = {best_ubcf_k}, RMSE = {ubcf_tune_df["RMSE"].min():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ubcf_tune_df['k'], ubcf_tune_df['RMSE'], 'o-', color='steelblue', lw=2)
axes[0].set_title('UB-CF: RMSE vs Neighborhood Size'); axes[0].set_xlabel('k'); axes[0].set_ylabel('RMSE')
axes[1].plot(ubcf_tune_df['k'], ubcf_tune_df['MAE'], 'o-', color='coral', lw=2)
axes[1].set_title('UB-CF: MAE vs Neighborhood Size'); axes[1].set_xlabel('k'); axes[1].set_ylabel('MAE')
plt.tight_layout(); plt.savefig('results/ubcf_tuning.png', bbox_inches='tight'); plt.show()

<a id='9'></a>
## 9. Top-N Candidate Generation

Generate top-20 unrated recipes per user. SVD uses chunked matrix reconstruction (500 users/chunk). UB-CF uses chunked similarity computation (200 users/chunk).

In [ ]:
# ── SVD Top-N ──
t0 = time.time()
Uf, sf, Vtf = svds(R_centered, k=best_svd_k)
Uf, sf, Vtf = Uf[:, ::-1], sf[::-1], Vtf[::-1, :]
Usf = Uf * sf

svd_rows = []
for start in range(0, n_users, 500):
    end = min(start + 500, n_users)
    Rp = np.clip(Usf[start:end] @ Vtf + user_means[start:end, None], 1, 5)
    Ra = R_sparse[start:end].toarray()
    Rp[Ra > 0] = -np.inf  # mask already-rated

    ti = np.argsort(-Rp, axis=1)[:, :TOP_N]
    ts = np.take_along_axis(Rp, ti, axis=1)

    for ul in range(end - start):
        for rk in range(TOP_N):
            if ts[ul, rk] == -np.inf:
                continue
            svd_rows.append({
                'user_id': idx2user[start + ul],
                'recipe_id': idx2recipe[ti[ul, rk]],
                'predicted_rating': round(float(ts[ul, rk]), 4),
                'rank': rk + 1})
    del Rp, Ra; gc.collect()

svd_cands = pd.DataFrame(svd_rows)
del svd_rows, Uf, sf, Vtf, Usf; gc.collect()
print(f'SVD candidates: {len(svd_cands):,} ({time.time() - t0:.0f}s)')

In [ ]:
# ── UB-CF Top-N ──
t0 = time.time()
rated_mask = (R_sparse > 0).astype(np.float64)
ubcf_rows = []

for start in range(0, n_users, 200):
    end = min(start + 200, n_users)
    if start % 1000 == 0:
        print(f'  Users {start}–{end} ({time.time() - t0:.0f}s)')

    sim_chunk = sk_cosine(R_centered[start:end], R_centered)
    for i in range(end - start):
        sim_chunk[i, start + i] = 0  # no self-similarity
    sim_chunk = np.maximum(sim_chunk, 0)

    num = sim_chunk @ R_centered.toarray()
    den = sim_chunk @ rated_mask.toarray()

    with np.errstate(invalid='ignore', divide='ignore'):
        Rp = np.where(den > 0, num / den, 0.0)
    Rp += user_means[start:end, None]
    Rp = np.clip(Rp, 1, 5)

    Ra = R_sparse[start:end].toarray()
    Rp[Ra > 0] = -np.inf

    ti = np.argsort(-Rp, axis=1)[:, :TOP_N]
    ts = np.take_along_axis(Rp, ti, axis=1)

    for ul in range(end - start):
        for rk in range(TOP_N):
            if ts[ul, rk] == -np.inf:
                continue
            ubcf_rows.append({
                'user_id': idx2user[start + ul],
                'recipe_id': idx2recipe[ti[ul, rk]],
                'predicted_rating': round(float(ts[ul, rk]), 4),
                'rank': rk + 1})

    del sim_chunk, num, den, Rp, Ra; gc.collect()

ubcf_cands = pd.DataFrame(ubcf_rows)
del ubcf_rows, rated_mask; gc.collect()
print(f'UB-CF candidates: {len(ubcf_cands):,} ({time.time() - t0:.0f}s)')

In [ ]:
# Save candidates
svd_cands.to_csv('data/processed/cf_svd_candidates.csv', index=False)
ubcf_cands.to_csv('data/processed/cf_ubased_candidates.csv', index=False)
print('Saved candidate lists.')

<a id='10'></a>
## 10. Popularity Baseline

Recommend the most-rated recipes (from training data) that the user hasn't already rated. This is a non-personalized baseline — useful for checking whether personalization actually helps.

In [ ]:
recipe_pop = train_df.groupby('r_idx').size().sort_values(ascending=False)
popular_order = recipe_pop.index.values

pop_rows = []
for u_idx in range(n_users):
    rated = set(R_sparse[u_idx].nonzero()[1])
    rank = 0
    for r_idx in popular_order:
        if r_idx not in rated:
            rank += 1
            pop_rows.append({
                'user_id': idx2user[u_idx],
                'recipe_id': idx2recipe[int(r_idx)],
                'predicted_rating': 0.0,
                'rank': rank})
            if rank >= TOP_N:
                break

pop_cands = pd.DataFrame(pop_rows)
print(f'Popularity baseline: {len(pop_cands):,} candidates')
print(f'Most popular recipe has {recipe_pop.iloc[0]:,} training ratings')

<a id='11'></a>
## 11. Ingredient Constraint Filtering

For each user we simulate a pantry: pool all ingredients from recipes they've rated, randomly keep 70%. Then filter candidates:
- **Hard:** 0 missing ingredients allowed
- **Soft:** up to `max_missing` ingredients missing, re-ranked by feasibility first

Six dietary restriction presets are also supported (used in the Streamlit app):

| Preset | Excludes |
|---|---|
| Halal-friendly | pork, bacon, ham, lard, etc. |
| No Red Meat | beef, lamb, veal, steak, etc. |
| Vegetarian | all meat and seafood |
| Vegan | meat, seafood, dairy, eggs, honey |
| Gluten-Free | flour, wheat, pasta, breadcrumbs, etc. |
| Dairy-Free | milk, butter, cheese, cream, yogurt, etc. |

In [ ]:
# Build ingredient lookup: recipe_id -> frozenset of ingredients
rc = recipes[['recipe_id', 'ingredients_str']].copy()
rc['ing_set'] = rc['ingredients_str'].apply(
    lambda x: frozenset(i.strip().lower() for i in str(x).split('|')) if pd.notna(x) else frozenset())
recipe_ingredients = dict(zip(rc['recipe_id'], rc['ing_set']))
print(f'Ingredient lookup for {len(recipe_ingredients):,} recipes')

In [ ]:
def simulate_pantry(user_id, idf, ri, coverage=0.7, seed=42):
    """Build a simulated pantry from a user's rating history."""
    rng = np.random.default_rng(seed + hash(user_id) % 10000)
    user_recipes = idf[idf['user_id'] == user_id]['recipe_id'].unique()
    all_ings = set()
    for rid in user_recipes:
        if rid in ri:
            all_ings.update(ri[rid])
    if not all_ings:
        return frozenset()
    al = list(all_ings)
    return frozenset(rng.choice(al, max(1, int(len(al) * coverage)), replace=False))


def count_missing(rid, pantry, ri):
    """Number of recipe ingredients not in pantry."""
    if rid not in ri:
        return np.inf
    return len(ri[rid] - pantry)


def apply_hard(cdf, pantry, ri):
    """Keep only recipes with zero missing ingredients."""
    d = cdf.copy()
    d['n_missing'] = d['recipe_id'].apply(lambda r: count_missing(r, pantry, ri))
    f = d[d['n_missing'] == 0].copy()
    f['rank'] = range(1, len(f) + 1)
    return f


def apply_soft(cdf, pantry, ri, max_missing=3):
    """Keep recipes within threshold, re-rank by feasibility then predicted rating."""
    d = cdf.copy()
    d['n_missing'] = d['recipe_id'].apply(lambda r: count_missing(r, pantry, ri))
    f = d[d['n_missing'] <= max_missing].copy()
    f = f.sort_values(['n_missing', 'predicted_rating'], ascending=[True, False])
    f['rank'] = range(1, len(f) + 1)
    return f


def process_all_users(cdf, idf, ri, max_missing=3, coverage=0.7):
    """Apply both constraint modes for every user."""
    hard_parts, soft_parts = [], []
    users = cdf['user_id'].unique()
    t0 = time.time()
    for i, uid in enumerate(users):
        if i % 2000 == 0 and i > 0:
            print(f'  {i}/{len(users)} ({time.time() - t0:.0f}s)')
        uc = cdf[cdf['user_id'] == uid]
        pantry = simulate_pantry(uid, idf, ri, coverage)
        h = apply_hard(uc, pantry, ri); h['user_id'] = uid; hard_parts.append(h)
        s = apply_soft(uc, pantry, ri, max_missing); s['user_id'] = uid; soft_parts.append(s)
    print(f'  Done ({time.time() - t0:.0f}s)')
    return (pd.concat(hard_parts, ignore_index=True) if hard_parts else pd.DataFrame(),
            pd.concat(soft_parts, ignore_index=True) if soft_parts else pd.DataFrame())

In [ ]:
print('=== SVD constraints ===')
svd_hard, svd_soft = process_all_users(svd_cands, interactions, recipe_ingredients, SOFT_MAX_MISSING)
print(f'  Hard: {len(svd_hard):,} | Soft: {len(svd_soft):,}')

print('\n=== UB-CF constraints ===')
ubcf_hard, ubcf_soft = process_all_users(ubcf_cands, interactions, recipe_ingredients, SOFT_MAX_MISSING)
print(f'  Hard: {len(ubcf_hard):,} | Soft: {len(ubcf_soft):,}')

# Save constrained results
svd_hard.to_csv('data/processed/constrained_svd_hard.csv', index=False)
svd_soft.to_csv('data/processed/constrained_svd_soft.csv', index=False)
ubcf_hard.to_csv('data/processed/constrained_ubcf_hard.csv', index=False)
ubcf_soft.to_csv('data/processed/constrained_ubcf_soft.csv', index=False)
print('\nSaved constrained candidate lists.')

In [ ]:
# Candidate coverage
cov_df = pd.DataFrame({
    'System': ['UB-CF Hard', 'UB-CF Soft', 'SVD Hard', 'SVD Soft'],
    'Surviving': [len(ubcf_hard), len(ubcf_soft), len(svd_hard), len(svd_soft)],
    'Total': [len(ubcf_cands), len(ubcf_cands), len(svd_cands), len(svd_cands)]})
cov_df['Coverage (%)'] = (cov_df['Surviving'] / cov_df['Total'] * 100).round(2)
print(cov_df.to_string(index=False))

plt.figure(figsize=(9, 4))
colors = ['#4C72B0', '#4C72B0', '#DD8452', '#DD8452']
bars = plt.bar(cov_df['System'], cov_df['Coverage (%)'], color=colors, edgecolor='white', width=0.5)
for b, v in zip(bars, cov_df['Coverage (%)']):
    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5, f'{v:.1f}%', ha='center', fontsize=11)
plt.title('Candidate Coverage by Constraint Mode')
plt.ylabel('% of candidates surviving'); plt.ylim(0, 110)
plt.tight_layout(); plt.savefig('results/csr_comparison.png', bbox_inches='tight'); plt.show()

In [ ]:
# Missing ingredient distribution (soft mode)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, title in [(axes[0], ubcf_soft, 'UB-CF Soft'), (axes[1], svd_soft, 'SVD Soft')]:
    if len(df) > 0:
        c = df['n_missing'].value_counts().sort_index()
        ax.bar(c.index, c.values, color='steelblue', edgecolor='white')
    ax.set_title(f'{title} — Missing Ingredients')
    ax.set_xlabel('Missing ingredients'); ax.set_ylabel('Recommendations')
plt.tight_layout(); plt.savefig('results/missing_ingredient_distribution.png', bbox_inches='tight'); plt.show()

<a id='12'></a>
## 12. Constraint Threshold Analysis

Sweep `max_missing` from 0 (hard) to 5. How does loosening the constraint affect Precision@10 and candidate coverage?

In [ ]:
# Ground truth: recipes each user liked in the test set
liked = (test_df[test_df['rating'] >= RELEVANCE_THRESHOLD]
         .groupby('user_id')['recipe_id'].apply(set).to_dict())

def prec_rec_at_k(cdf, liked_dict, k):
    """Mean Precision@K and Recall@K across users."""
    ps, rs = [], []
    for uid, g in cdf.groupby('user_id'):
        topk = g.nsmallest(k, 'rank')['recipe_id'].values
        rel = liked_dict.get(uid, set())
        if not rel:
            continue
        hits = sum(1 for r in topk if r in rel)
        ps.append(hits / k)
        rs.append(hits / len(rel))
    return (np.mean(ps) if ps else 0, np.mean(rs) if rs else 0)

In [ ]:
thresh_results = []

for mm in [0, 1, 2, 3, 4, 5]:
    for model_name, cands in [('SVD', svd_cands), ('UB-CF', ubcf_cands)]:
        soft_parts = []
        for uid in cands['user_id'].unique():
            uc = cands[cands['user_id'] == uid]
            pantry = simulate_pantry(uid, interactions, recipe_ingredients, PANTRY_COVERAGE)
            s = apply_soft(uc, pantry, recipe_ingredients, mm)
            s['user_id'] = uid
            soft_parts.append(s)
        con = pd.concat(soft_parts, ignore_index=True) if soft_parts else pd.DataFrame()

        coverage = len(con) / len(cands) * 100 if len(cands) else 0
        p10, r10 = prec_rec_at_k(con, liked, 10) if len(con) else (0, 0)

        thresh_results.append({
            'Model': model_name, 'max_missing': mm,
            'Coverage (%)': round(coverage, 2),
            'P@10': round(p10, 6), 'R@10': round(r10, 6)})
    print(f'max_missing = {mm} done')

thresh_df = pd.DataFrame(thresh_results)
print(thresh_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for model, color in [('SVD', '#DD8452'), ('UB-CF', '#4C72B0')]:
    s = thresh_df[thresh_df['Model'] == model]
    axes[0].plot(s['max_missing'], s['P@10'], 'o-', color=color, label=model, lw=2)
    axes[1].plot(s['max_missing'], s['Coverage (%)'], 'o-', color=color, label=model, lw=2)
    axes[2].plot(s['Coverage (%)'], s['P@10'], 'o-', color=color, label=model, lw=2)

axes[0].set_title('Precision@10 vs Max Missing'); axes[0].set_xlabel('max_missing'); axes[0].set_ylabel('P@10'); axes[0].legend()
axes[1].set_title('Coverage vs Max Missing'); axes[1].set_xlabel('max_missing'); axes[1].set_ylabel('Coverage (%)'); axes[1].legend()
axes[2].set_title('Precision vs Coverage (Trade-off)'); axes[2].set_xlabel('Coverage (%)'); axes[2].set_ylabel('P@10'); axes[2].legend()

plt.tight_layout(); plt.savefig('results/constraint_threshold_analysis.png', bbox_inches='tight'); plt.show()

<a id='13'></a>
## 13. Evaluation

Three metric families:
1. **Rating Prediction:** RMSE and MAE on the held-out test set
2. **Top-N Quality:** Precision@K and Recall@K at K = 5, 10, 20
3. **Constraint Satisfaction Rate (CSR)**

### 13a. Rating Prediction Accuracy

In [ ]:
# Baselines
global_mean = train_df['rating'].mean()
gm_rmse = np.sqrt(np.mean((test_mapped['rating'].values - global_mean) ** 2))
gm_mae  = np.mean(np.abs(test_mapped['rating'].values - global_mean))

um_preds = user_means[test_mapped['u_idx'].values]
um_rmse = np.sqrt(np.mean((test_mapped['rating'].values - um_preds) ** 2))
um_mae  = np.mean(np.abs(test_mapped['rating'].values - um_preds))

acc_df = pd.DataFrame({
    'Model': ['Global Mean', 'User Mean', 'User-Based CF', 'SVD'],
    'RMSE':  [gm_rmse, um_rmse, ubcf_rmse, svd_rmse],
    'MAE':   [gm_mae,  um_mae,  ubcf_mae,  svd_mae]})
print(acc_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#c7c7c7', '#aec7e8', '#4C72B0', '#DD8452']
for ax, m in zip(axes, ['RMSE', 'MAE']):
    bars = ax.bar(acc_df['Model'], acc_df[m], color=colors, edgecolor='white', width=0.5)
    for b, v in zip(bars, acc_df[m]):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.002, f'{v:.4f}', ha='center', fontsize=10)
    ax.set_title(f'{m} — Rating Prediction'); ax.set_ylabel(m)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.savefig('results/rmse_mae.png', bbox_inches='tight'); plt.show()

### 13b. Precision@K and Recall@K

In [ ]:
systems = {
    'Popularity': pop_cands, 'UB-CF': ubcf_cands, 'SVD': svd_cands,
    'UB-CF + Hard': ubcf_hard, 'UB-CF + Soft': ubcf_soft,
    'SVD + Hard': svd_hard, 'SVD + Soft': svd_soft}

pk_rows = []
for name, df in systems.items():
    if len(df) == 0:
        continue
    row = {'System': name}
    for k in K_VALUES:
        p, r = prec_rec_at_k(df, liked, k)
        row[f'P@{k}'] = round(p, 6)
        row[f'R@{k}'] = round(r, 6)
    pk_rows.append(row)

pk_df = pd.DataFrame(pk_rows)
print(pk_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette('tab10', len(systems))

for ax, pf, title in [(axes[0], 'P@', 'Precision@K'), (axes[1], 'R@', 'Recall@K')]:
    for i, name in enumerate(systems):
        row = pk_df[pk_df['System'] == name]
        if len(row) == 0:
            continue
        vals = [row[f'{pf}{k}'].values[0] for k in K_VALUES]
        ax.plot(K_VALUES, vals, 'o-', label=name, color=palette[i], lw=2)
    ax.set_title(f'{title} vs K'); ax.set_xlabel('K'); ax.set_ylabel(title)
    ax.legend(fontsize=8); ax.set_xticks(K_VALUES)

plt.tight_layout(); plt.savefig('results/precision_recall_at_k.png', bbox_inches='tight'); plt.show()

### 13c. Heatmap

In [ ]:
hcols = [f'P@{k}' for k in K_VALUES] + [f'R@{k}' for k in K_VALUES]
hdata = pk_df.set_index('System')[hcols].astype(float)

plt.figure(figsize=(10, 5))
sns.heatmap(hdata, annot=True, fmt='.4f', cmap='YlOrRd', linewidths=0.5)
plt.title('Precision@K and Recall@K — All Systems')
plt.tight_layout(); plt.savefig('results/metrics_heatmap.png', bbox_inches='tight'); plt.show()

### 13d. Constraint Satisfaction Rate

In [ ]:
def compute_csr(df, mode='hard', mm=3):
    if 'n_missing' not in df.columns:
        return None
    if mode == 'hard':
        return (df['n_missing'] == 0).mean() * 100
    return (df['n_missing'] <= mm).mean() * 100

csr_rows = []
for name, df in [('UB-CF + Hard', ubcf_hard), ('UB-CF + Soft', ubcf_soft),
                  ('SVD + Hard', svd_hard), ('SVD + Soft', svd_soft)]:
    csr_rows.append({
        'System': name,
        'Hard CSR': compute_csr(df, 'hard'),
        'Soft CSR': compute_csr(df, 'soft', SOFT_MAX_MISSING)})

csr_df = pd.DataFrame(csr_rows)
print(csr_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(csr_df)); w = 0.35
b1 = ax.bar(x - w/2, csr_df['Hard CSR'], w, label='Hard (0 missing)', color='steelblue')
b2 = ax.bar(x + w/2, csr_df['Soft CSR'], w, label=f'Soft (≤{SOFT_MAX_MISSING})', color='#DD8452')
for bars in [b1, b2]:
    for b in bars:
        if b.get_height() > 0:
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f'{b.get_height():.1f}%', ha='center', fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(csr_df['System'])
ax.set_ylabel('CSR (%)'); ax.set_title('Constraint Satisfaction Rate'); ax.legend(); ax.set_ylim(0, 115)
plt.tight_layout(); plt.savefig('results/constraint_satisfaction_rate.png', bbox_inches='tight'); plt.show()

### 13e. Trade-off: Quality vs Feasibility

In [ ]:
plt.figure(figsize=(9, 5))
for name, df in systems.items():
    if 'n_missing' not in df.columns:
        continue
    p10, _ = prec_rec_at_k(df, liked, 10)
    csr_val = compute_csr(df, 'soft', SOFT_MAX_MISSING)
    plt.scatter(csr_val, p10, s=80, color='#DD8452', zorder=5)
    plt.annotate(name, (csr_val, p10), textcoords='offset points', xytext=(8, 4), fontsize=9)

plt.xlabel('Constraint Satisfaction Rate (%)'); plt.ylabel('Precision@10')
plt.title('Trade-off: Recommendation Quality vs Feasibility')
plt.tight_layout(); plt.savefig('results/tradeoff_precision_csr.png', bbox_inches='tight'); plt.show()

### 13f. Complete Results Table

In [ ]:
final_rows = []
for name, df in systems.items():
    row = {'System': name}

    if name == 'UB-CF':
        row['RMSE'] = round(ubcf_rmse, 4); row['MAE'] = round(ubcf_mae, 4)
    elif name == 'SVD':
        row['RMSE'] = round(svd_rmse, 4); row['MAE'] = round(svd_mae, 4)
    else:
        row['RMSE'] = '—'; row['MAE'] = '—'

    pr = pk_df[pk_df['System'] == name]
    if len(pr) > 0:
        for k in K_VALUES:
            row[f'Precision@{k}'] = pr[f'P@{k}'].values[0]
            row[f'Recall@{k}'] = pr[f'R@{k}'].values[0]

    if 'n_missing' in df.columns:
        row['CSR'] = f"{compute_csr(df, 'soft', SOFT_MAX_MISSING):.1f}%"
    else:
        row['CSR'] = 'N/A'
    final_rows.append(row)

final_df = pd.DataFrame(final_rows)
final_df.to_csv('results/metrics_comparison.csv', index=False)
print(final_df.to_string(index=False))

<a id='14'></a>
## 14. User-Stratified Analysis

Do active users get better recommendations than casual ones? Split at the median training-set rating count.

In [ ]:
ua = train_df.groupby('user_id').size().reset_index(name='n_train')
med = ua['n_train'].median()
high_u = set(ua[ua['n_train'] >= med]['user_id'])
low_u  = set(ua[ua['n_train'] <  med]['user_id'])
print(f'Median training ratings: {med:.0f}')
print(f'High-activity: {len(high_u):,} | Low-activity: {len(low_u):,}')

In [ ]:
strat = []
for activity, uset in [('High', high_u), ('Low', low_u)]:
    for name, df in [('Popularity', pop_cands), ('UB-CF', ubcf_cands),
                      ('SVD', svd_cands), ('SVD + Soft', svd_soft)]:
        sub = df[df['user_id'].isin(uset)]
        if len(sub) == 0:
            continue
        p5, r5   = prec_rec_at_k(sub, liked, 5)
        p10, r10 = prec_rec_at_k(sub, liked, 10)
        strat.append({
            'Activity': activity, 'System': name,
            'P@5': round(p5, 6), 'P@10': round(p10, 6),
            'R@5': round(r5, 6), 'R@10': round(r10, 6)})

strat_df = pd.DataFrame(strat)
print(strat_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in [(axes[0], 'P@10', 'Precision@10'), (axes[1], 'R@10', 'Recall@10')]:
    sn = strat_df['System'].unique()
    x = np.arange(len(sn)); w = 0.35

    hv = [strat_df[(strat_df['System'] == s) & (strat_df['Activity'] == 'High')][metric].values for s in sn]
    lv = [strat_df[(strat_df['System'] == s) & (strat_df['Activity'] == 'Low')][metric].values for s in sn]
    hv = [v[0] if len(v) else 0 for v in hv]
    lv = [v[0] if len(v) else 0 for v in lv]

    ax.bar(x - w/2, hv, w, label='High activity', color='steelblue')
    ax.bar(x + w/2, lv, w, label='Low activity', color='coral')
    ax.set_xticks(x); ax.set_xticklabels(sn, rotation=15)
    ax.set_title(f'{title} by User Activity'); ax.set_ylabel(metric); ax.legend()

plt.tight_layout(); plt.savefig('results/user_stratified_analysis.png', bbox_inches='tight'); plt.show()

## Sample Outputs — Worked Example

Walk through the full pipeline for a single user to show what each stage produces.

### User Profile

In [ ]:
DEMO_USER = 1634

user_hist = interactions[interactions['user_id'] == DEMO_USER]
user_recipes = user_hist.merge(
    recipes[['recipe_id', 'name', 'n_ingredients', 'minutes']], on='recipe_id', how='left')

print(f'User {DEMO_USER}')
print(f'  Ratings: {len(user_hist)}  |  Average: {user_hist["rating"].mean():.2f}  |  Range: {user_hist["rating"].min()}–{user_hist["rating"].max()}')
print(f'\nRating distribution:')
print(user_hist['rating'].value_counts().sort_index().to_frame('count').to_string())
print(f'\nTop-rated recipes:')
print(user_recipes.sort_values('rating', ascending=False).head(10)[['name','rating','minutes','n_ingredients']].to_string(index=False))

### SVD Top-10 (Unconstrained)

In [ ]:
demo_u_idx = user2idx[DEMO_USER]

# Recompute SVD predictions for this user
Ud, sd, Vtd = svds(R_centered, k=best_svd_k)
Ud, sd, Vtd = Ud[:,::-1], sd[::-1], Vtd[::-1,:]

svd_pred_row = np.clip((Ud[demo_u_idx] * sd) @ Vtd + user_means[demo_u_idx], 1.0, 5.0)
demo_rated = set(R_sparse[demo_u_idx].nonzero()[1])
svd_pred_masked = svd_pred_row.copy()
svd_pred_masked[list(demo_rated)] = -np.inf

svd_top10_idx = np.argsort(svd_pred_masked)[::-1][:10]
svd_demo = []
for rank, ri in enumerate(svd_top10_idx, 1):
    rid = idx2recipe[ri]
    nm = recipes.loc[recipes['recipe_id']==rid, 'name'].values
    svd_demo.append({'Rank': rank, 'Recipe': nm[0].title() if len(nm) else str(rid),
                     'Predicted': round(float(svd_pred_masked[ri]), 3)})

print(f'SVD Top-10 for User {DEMO_USER}:')
print(pd.DataFrame(svd_demo).to_string(index=False))
del Ud, sd, Vtd; gc.collect()

### UB-CF Top-10 (Unconstrained)

In [ ]:
sim = sk_cosine(R_centered[demo_u_idx], R_centered).flatten()
sim[demo_u_idx] = 0; sim = np.maximum(sim, 0)

rm_dense = (R_sparse > 0).astype(np.float64).toarray()
num = sim @ R_centered.toarray()
den = sim @ rm_dense
with np.errstate(invalid='ignore', divide='ignore'):
    ubcf_pred_row = np.where(den > 0, num / den, 0.0)
ubcf_pred_row = np.clip(ubcf_pred_row + user_means[demo_u_idx], 1.0, 5.0)
ubcf_pred_row[list(demo_rated)] = -np.inf

ubcf_top10_idx = np.argsort(ubcf_pred_row)[::-1][:10]
ubcf_demo = []
for rank, ri in enumerate(ubcf_top10_idx, 1):
    rid = idx2recipe[ri]
    nm = recipes.loc[recipes['recipe_id']==rid, 'name'].values
    ubcf_demo.append({'Rank': rank, 'Recipe': nm[0].title() if len(nm) else str(rid),
                      'Predicted': round(float(ubcf_pred_row[ri]), 3)})

print(f'UB-CF Top-10 for User {DEMO_USER}:')
print(pd.DataFrame(ubcf_demo).to_string(index=False))
del rm_dense, num, den; gc.collect()

### Constraint Filtering — Hard vs Soft

Simulate this user's pantry (70% of ingredients from their rated recipes), then show which SVD candidates survive each constraint mode.

In [ ]:
# Simulate pantry
user_rids = interactions[interactions['user_id']==DEMO_USER]['recipe_id'].unique()
ri_map = {}
for _, r in recipes.iterrows():
    ings = r.get('ingredients', '')
    ri_map[r['recipe_id']] = frozenset(i.strip().lower() for i in str(ings).split('|')) if pd.notna(ings) else frozenset()

demo_all_ings = set()
for rid in user_rids:
    if rid in ri_map: demo_all_ings.update(ri_map[rid])

rng = np.random.default_rng(42 + hash(DEMO_USER) % 10000)
nk = max(1, int(len(demo_all_ings) * 0.7))
demo_pantry = frozenset(rng.choice(list(demo_all_ings), size=nk, replace=False))

print(f'Pantry: {len(demo_pantry)} ingredients (70% of {len(demo_all_ings)} from history)')
print(f'Sample: {sorted(list(demo_pantry))[:12]}...')

# Filter SVD top-20
top20_idx = np.argsort(svd_pred_masked)[::-1][:20]
print(f'\n{"Recipe":<42} {"Pred":>6} {"Miss":>5}  Result')
print('─' * 75)
n_hard = n_soft = 0
for ri in top20_idx:
    rid = idx2recipe[ri]
    nm = recipes.loc[recipes['recipe_id']==rid, 'name'].values
    nm = nm[0].title()[:40] if len(nm) else str(rid)
    pred = svd_pred_masked[ri]
    missing = ri_map.get(rid, frozenset()) - demo_pantry
    n_miss = len(missing)
    if n_miss == 0:
        tag = '✓ HARD + SOFT'; n_hard += 1; n_soft += 1
    elif n_miss <= 3:
        tag = f'~ SOFT only ({", ".join(list(missing)[:3])})'; n_soft += 1
    else:
        tag = f'✗ filtered ({n_miss} missing)'
    print(f'{nm:<42} {pred:>6.3f} {n_miss:>5}  {tag}')

print(f'\nHard survivors: {n_hard}/20  |  Soft survivors (≤3): {n_soft}/20')

### Actual vs Predicted — Test Set

In [ ]:
user_test = test_mapped[test_mapped['u_idx'] == demo_u_idx].copy()
if len(user_test) > 0:
    rows = []
    for _, row in user_test.iterrows():
        rid = row['recipe_id']
        nm = recipes.loc[recipes['recipe_id']==rid, 'name'].values
        nm = nm[0].title() if len(nm) else str(rid)
        rows.append({
            'Recipe': nm,
            'Actual': int(row['rating']),
            'SVD': round(row['svd_pred'], 2) if pd.notna(row.get('svd_pred')) else '—',
            'UB-CF': round(row['ubcf_pred'], 2) if pd.notna(row.get('ubcf_pred')) else '—',
        })
    print(f'User {DEMO_USER} — Test Set: Actual vs Predicted')
    print(pd.DataFrame(rows).to_string(index=False))
    svd_err = np.sqrt(np.mean((user_test['rating'] - user_test['svd_pred'])**2))
    print(f'\nSVD RMSE on this user: {svd_err:.3f}')
    if 'ubcf_pred' in user_test.columns and user_test['ubcf_pred'].notna().any():
        vm = user_test['ubcf_pred'].notna()
        print(f'UB-CF RMSE on this user: {np.sqrt(np.mean((user_test.loc[vm,"rating"] - user_test.loc[vm,"ubcf_pred"])**2)):.3f}')
else:
    print(f'User {DEMO_USER} has no mapped test entries.')

### Popularity vs SVD — Side by Side

In [ ]:
pop_order = train_df.groupby('r_idx').size().sort_values(ascending=False).index.values
pop_top10 = []
rank = 0
for ri in pop_order:
    if ri not in demo_rated:
        rank += 1
        rid = idx2recipe[int(ri)]
        nm = recipes.loc[recipes['recipe_id']==rid, 'name'].values
        pop_top10.append({'Rank': rank, 'Recipe': nm[0].title() if len(nm) else str(rid)})
        if rank >= 10: break

print('POPULARITY (most rated):')
print(pd.DataFrame(pop_top10).to_string(index=False))
print()
print('SVD (personalized):')
print(pd.DataFrame(svd_demo)[['Rank','Recipe']].to_string(index=False))

overlap = set(r['Recipe'] for r in pop_top10) & set(r['Recipe'] for r in svd_demo)
print(f'\nOverlap: {len(overlap)} recipes in both lists')

### Constraint Threshold Effect — Single User

In [ ]:
top100_idx = np.argsort(svd_pred_masked)[::-1][:100]
cands = [(idx2recipe[ri], float(svd_pred_masked[ri]),
          len(ri_map.get(idx2recipe[ri], frozenset()) - demo_pantry))
         for ri in top100_idx]

print(f'User {DEMO_USER} — SVD top-100 candidates surviving at each threshold:')
print(f'{"max_missing":>12} {"Surviving":>10} {"Avg Predicted":>14}')
print('─' * 40)
for mm in [0, 1, 2, 3, 4, 5]:
    alive = [(r, p, m) for r, p, m in cands if m <= mm]
    if alive:
        print(f'{mm:>12} {len(alive):>10} {np.mean([p for _,p,_ in alive]):>14.3f}')
    else:
        print(f'{mm:>12} {0:>10} {"—":>14}')

### Dataset Summary

In [ ]:
print('=== Dataset ===')
print(f'  After preprocessing: {len(interactions):,} interactions | {n_users:,} users | {n_recipes:,} recipes')
print(f'  Sparsity: {sparsity:.4%}')
print(f'  Train: {len(train_df):,} | Test: {len(test_df):,}')
print()
print('=== Rating Prediction (Test Set) ===')
print(f'  {"Model":<20} {"RMSE":>8} {"MAE":>8}')
print(f'  {"─"*38}')
print(f'  {"Global Mean":<20} {gm_rmse:>8.4f} {gm_mae:>8.4f}')
print(f'  {"User Mean":<20} {um_rmse:>8.4f} {um_mae:>8.4f}')
print(f'  {"UB-CF":<20} {ubcf_rmse:>8.4f} {ubcf_mae:>8.4f}')
print(f'  {"SVD":<20} {svd_rmse:>8.4f} {svd_mae:>8.4f}')
print()
print('=== Tuning Results ===')
print('SVD:')
print(svd_tune_df.to_string(index=False))
print('\nUB-CF:')
print(ubcf_tune_df.to_string(index=False))

<a id='15'></a>
## 15. Summary

**SVD outperforms UB-CF across all metrics.** It achieves the best RMSE and highest Precision@K and Recall@K at every list length and under both constraint modes. This holds after tuning UB-CF's neighborhood size.

**UB-CF underperforms the global mean baseline on RMSE.** Sweeping neighborhood size from 10 to 200 made little difference. At >99% sparsity with heavy positive skew, the neighbor average introduces more noise than signal. Its MAE still beats the baseline, so it captures preference direction even when magnitude is off.

**SVD is stable across factor counts.** RMSE barely moves from k=10 to k=150. The first 10–20 factors capture most of the usable signal; the scree plot confirms variance grows linearly with no clear elbow.

**Soft constraints are more practical.** Hard mode kills 99%+ of candidates — not enough left to recommend. Soft with max_missing=3 retains 18–29% of candidates at full constraint satisfaction.

**Returns diminish past max_missing=3.** The threshold sweep shows coverage increasing but precision flattening, making 3 a reasonable default.

**Popularity baseline is competitive on Precision@K** due to the dataset's popularity bias. SVD's advantage shows up more clearly in Recall@K, where it surfaces niche items the user would actually like.

**High-activity users benefit more from personalization.** Users with many ratings give CF models enough signal to outperform the popularity baseline. Low-activity users see smaller gains.